# Part 6 of 6: Results Summary & Discussion

COMP7015 Group Project – Topic 1: Plant Disease Classification. This notebook collects the key tables of notebooks 01–05 in one place (Section 21) and holds the discussion (Section 22).

The notebooks run in order: `01_data` → `02_cnn_resnet` → `03_vit` → `04_test_evaluation` → `05_robustness` → `06_discussion`. The shared code lives in the `plantdisease/` package. Finished training runs and predictions are saved under `OUT_DIR/runs/` and reused, so this notebook can be re-run on its own once the earlier ones have run.

In [ ]:
import os

import pandas as pd
from IPython.display import Markdown, display

from plantdisease.config import out_path

TABLES = [
    ("Data: class counts per split", "class_counts.csv"),
    ("CNN / ResNet18 tuning and ablations, seed 62 (Section 9)", "tuning_results.csv"),
    ("CNN / ResNet18 selection with all seeds (Section 9.3)", "tuning_multi_seed.csv"),
    ("ViT strategies: validation and cost (Section 10.3)", "vit_validation.csv"),
    ("ViT strategies: test (Section 10.4)", "vit_strategies_test.csv"),
    (
        "ViT strategies: same-seed comparisons (Section 10.4)",
        "vit_strategy_comparison.csv",
    ),
    ("Final models on the test set (Section 11)", "test_results.csv"),
    (
        "Final models on the test set, seed by seed (Section 11)",
        "test_results_per_seed.csv",
    ),
    ("Compute trade-offs (Section 12)", "compute_tradeoffs.csv"),
    ("Errors that every seed makes (Section 13.1)", "stable_errors.csv"),
    ("Background alone (Section 14.1)", "background_only.csv"),
    ("Robustness on the same test leaves (Section 14)", "robustness.csv"),
    ("ViT retrained on other inputs (Section 15)", "input_training_grid.csv"),
    ("Same leaf, other background (Section 16)", "background_swap_tests.csv"),
    (
        "Same leaf, other background: paired and per background class (Section 16)",
        "background_swap_paired.csv",
    ),
    ("Attention on the leaf (Sections 17, 18.4)", "leaf_attention_countermeasure.csv"),
    ("Countermeasure (Section 18.3)", "countermeasure_results.csv"),
    (
        "Countermeasure vs. baseline, seed by seed (Section 18.3)",
        "countermeasure_comparison.csv",
    ),
    (
        "Countermeasure: predictions that follow the leaf or the background (Section 18.3)",
        "countermeasure_background_following.csv",
    ),
    ("PlantDoc field photos (Section 19.1)", "field_photos.csv"),
    (
        "PlantDoc field photos: recall per class (Section 19.1)",
        "field_photos_recall.csv",
    ),
    ("Confidence and abstention (Section 20)", "confidence_abstention.csv"),
]

## 21. Results Summary

Every table below is read from `OUT_DIR`, where notebooks 01–05 save them. A missing table means that its notebook has not run yet.

In [ ]:
for title, file_name in TABLES:
    display(Markdown(f"**{title}** (`{file_name}`)"))
    path = out_path(file_name)
    if os.path.exists(path):
        display(pd.read_csv(path))
    else:
        display(Markdown("*not available yet*"))

## 22. Discussion

Unless stated otherwise, scores are the macro-F1 on the 688 test photos, given as mean ± sd over the three training seeds. "Higher in k of 3 seeds" compares two models trained with the same seeds. All numbers come from the tables in Section 21.

**In short.** On clean PlantVillage photos all three models are almost perfect (macro-F1 0.991–0.997), and they differ by only a few photos. This score says little about the models outside the training conditions. The background alone predicts the class, the custom CNN largely follows it, and every model fails on field photos. Training with random backgrounds removes the background shortcut of ResNet18 at no measurable cost on clean photos. It does not make the models transfer to the field.

### 22.1 Data protocol (notebook 01)

- Grouping by leaf leaves 4,626 photos of 1,157 physical leaves: 3,238 / 700 / 688 photos for train / validation / test. No leaf id crosses two splits. Only one photo was excluded, an early-blight photo without leaf information. Spider mites have about 1.7× more photos than each other class, which the CNNs balance with class weights.
- **The background alone predicts the class.** A random forest that sees only the mean colour of the four image corners reaches 88.1% accuracy (macro-F1 0.868) on the validation photos. Uniform guessing gives 25%, and always answering spider mites gives 36% (macro-F1 0.132). The mean border brightness differs little between classes (132–141 grey levels), so the class is given away by the colour of the background rather than by its brightness alone. Because the backgrounds differ between the classes, a model can reach a high clean score without looking at the disease. Notebook 05 therefore tests the background in several ways.

### 22.2 Models and tuning (notebooks 02, 03)

- **The validation set is saturated.** Almost every recipe reaches a validation macro-F1 of 0.99–1.00, and one validation error moves the accuracy by 0.0014. On seed 62 the SE-CNN ablations differ from the full recipe (1.000) by at most 0.010: without SE attention 0.990, without class weights 0.993, batch size 64 0.993, without augmentation 0.994, without dropout 0.996. The full recipe itself varies by ±0.010 over the three seeds (0.994 ± 0.010). **None of these single-seed ablations can therefore be separated from the seed-to-seed variation.** Re-running the finalists with all seeds mattered: the full recipe, best on seed 62, has the lowest mean over the seeds. The SE-CNN with lr = 3e-4 was selected as the most stable (0.998 ± 0.002).
- **ResNet18.** Every recipe that fine-tunes the backbone reaches 1.000 on seed 62. Only freezing the backbone costs clearly (0.976), the one ablation effect well above the seed noise. Full fine-tuning with a constant learning rate was selected (1.000 ± 0.000 over the seeds). Two-stage fine-tuning with a cosine schedule is practically tied (0.999 ± 0.001).
- **ViT strategies.** Unfreezing more of the ViT improves the test score in every seed. Head only reaches 0.986 ± 0.003, the last two blocks 0.994 ± 0.001 and full fine-tuning 0.997 ± 0.003. Partial minus head only is +0.008 ± 0.003 and full minus partial is +0.003 ± 0.002, both higher in 3 of 3 seeds. The number of test photos that every seed misclassifies falls from 7 to 3 to 0. The price is mainly memory. Trainable parameters grow from 3 k to 14.2 M to 85.8 M, peak memory from 701 to 895 to 2,556 MiB (3.6×), and the time per epoch doubles (15.7 → 31.5 s). The total training time grows by only 25% (235 → 295 s). Full fine-tuning reaches its best epoch after 4–5 epochs and stops early after about 9, while the head-only runs use all 15 epochs.
- **Augmentation** has no measurable effect on the clean test for the partial ViT: +0.0003 ± 0.0018, higher in 2 of 3 seeds. The test leaves look like the training leaves, so there is little for it to fix. Whether it helps under the changes of notebook 05 was not tested separately.

### 22.3 Clean test, compute and errors (notebook 04)

| | SE-CNN (lr 3e-4) | ResNet18 (full FT) | ViT-B/16 (full FT) |
|---|---|---|---|
| test macro-F1 | 0.991 ± 0.002 | 0.993 ± 0.001 | 0.997 ± 0.003 |
| errors per seed (of 688) | 7, 5, 7 | 4, 6, 5 | 4, 0, 3 |
| parameters / checkpoint size | 0.41 M / 1.7 MB | 11.2 M / 44.8 MB | 85.8 M / 343 MB |
| latency batch 1 / per image in batches of 64 | 1.1 / 0.36 ms | 3.1 / 0.38 ms | 8.5 / 2.57 ms |
| training time / peak memory | 142 s / 848 MiB | 59 s / 595 MiB | 295 s / 2,556 MiB |

- **Ranking.** The order is ViT > ResNet18 > SE-CNN, but the gaps are small: on average 6.3, 5.0 and 2.3 wrong photos per seed. Seed by seed, the ViT beats ResNet18 in 2 of 3 seeds and ResNet18 beats the SE-CNN in 2 of 3 seeds. The ViT alone ranges from 0 to 4 errors. On clean photos the three models are therefore **practically tied**. The ViT's lead fits the consistent gains of Section 10, but this test alone does not establish it.
- **Confusions.** Healthy is recognised almost perfectly (F1 ≥ 0.999 for every model). Most errors lie between **early blight and spider mites**: 10 of 19 pooled errors of the SE-CNN, 12 of 15 of ResNet18 and 4 of 7 of the ViT. The SE-CNN also mistakes leaf mold for spider mites (5) and for early blight (3). Wrong predictions are less confident than correct ones: mean confidence 0.72, 0.85 and 0.80, against at least 0.988 for correct predictions. Section 20 builds on this.
- **Stable errors.** Only 1 (SE-CNN), 3 (ResNet18) and 0 (ViT) test photos are misclassified by every seed, while 13, 7 and 6 photos are wrong in at least one seed. Most errors therefore depend on the seed. There is little sign of mislabelled test photos; the errors are borderline photos that tip one way or the other with the training randomness.
- **Compute.** The SE-CNN is 27× smaller than ResNet18 and the fastest on single images (1.1 ms). It still needs more epochs (24.7 ± 7.8) and more training memory than ResNet18, probably because the activations of its full-resolution feature maps, not its 0.4 M weights, dominate the memory. ResNet18 is the cheapest to train (59 s, 595 MiB) and in batches as fast as the SE-CNN. Compared with ResNet18, the ViT gains 0.0035 macro-F1 for 7.7× the parameters, 5× the training time, 4.3× the memory and 2.8× the single-image latency (6.7× in batches). At 343 MB and 8.5 ms per photo it suits a server better than a phone. All times come from one CUDA GPU with mixed precision and are comparable only within this table.

### 22.4 Robustness (notebook 05)

**Changes to the same test leaves (Section 14).**
- The SE-CNN, trained from scratch, collapses under every change (0.16–0.48). This includes a moderate darkening (×0.6) that lies outside its ±10% brightness jitter. With ImageNet pre-training, ResNet18 and the ViT are robust to darkening (0.977, 0.993), and the ViT also to noise (0.985). Blur hurts every model (0.35–0.67). Grayscale hurts the ViT (0.489) more than ResNet18 (0.706).
- The seed-to-seed variation grows from ±0.001–0.003 on clean photos to ±0.175 (ResNet18, blur) and ±0.136 (ResNet18, noise). Seeds that are equally good on clean photos can differ widely under these changes, so the clean score does not predict robustness.
- **Retraining on the changed input (Section 15) removes almost the whole drop.** A ViT trained and tested on grayscale reaches 0.987 ± 0.004, against 0.997 in colour. Colour is thus worth only about 0.01 macro-F1, and the grayscale drop to 0.489 is almost entirely a mismatch between training and test input. A ViT trained and tested on background-removed photos reaches 0.998 ± 0.002, as high as the colour model on original photos. **The leaf alone carries enough information; the background is a shortcut, not a necessity.** Each retrained model transfers poorly to the other inputs, e.g. trained on grayscale and tested on background-removed photos 0.660 ± 0.102.

**Leaf or background? (Sections 16, 17).**
- With exactly the same leaf pixels, a background of another class lowers the score of every model in every seed. The SE-CNN falls from 0.897 to 0.381, ResNet18 from 0.976 to 0.761 and the ViT from 0.922 to 0.771 (paired differences −0.52, −0.21 and −0.15, lower in 3 of 3 seeds). The SE-CNN predicts the class of the background photo for 45% of the leaves, ResNet18 for 16% and the ViT for 12%.
- The pull is strongest for **healthy backgrounds**. On a background from a healthy photo, a diseased leaf is called healthy in 62% of the cases by the SE-CNN and in 37% by the ViT. On spider-mite backgrounds the shares are 36% and 2%.
- The same-class control already costs something: 0.991 → 0.897 (SE-CNN), 0.993 → 0.976 (ResNet18), 0.997 → 0.922 (ViT). Cut-out leaves on tiled backgrounds are unfamiliar input in themselves. The background effect is the further drop from the same-class to the other-class background.
- **Grad-CAM (Section 17).** The leaf covers 38% of the image. The SE-CNN puts 40% of its Grad-CAM weight on the leaf, no more than a map that ignores the leaf. ResNet18 puts 56% there (60% on other-class backgrounds). This agrees with the background test: the SE-CNN looks at the whole photo, ResNet18 more at the leaf.

**Countermeasure (Section 18).**
- **ResNet18 + random backgrounds** shows no measurable clean cost: 0.991 ± 0.003 against 0.993 (−0.002 ± 0.003, higher in 1 of 3 seeds). Unseen striped backgrounds go from 0.305 to 0.969 (+0.66) and other-class backgrounds from 0.761 to 0.986 (+0.22), in 3 of 3 seeds. Background removed goes from 0.730 to 0.957, but black is a training background for this model. The share of predictions that follow the background falls from 15.6% to 0.5%, and the Grad-CAM leaf share rises from 0.56 to 0.66. **For ResNet18, random backgrounds remove the background shortcut without costing clean accuracy.**
- **SE-CNN + random backgrounds** also depends less on the background. Other-class backgrounds go from 0.381 to 0.708 and stripes from 0.217 to 0.629, background-following falls from 45% to 16%, and the leaf share rises from 0.40 to 0.59. But the clean score falls from 0.991 to 0.928 (−0.063 ± 0.010, lower in all seeds). Without the shortcut, the small model trained from scratch has to learn the disease from 3,238 photos. With the same epochs and early stopping it does not reach the baseline's clean score.
- **The class-matched control** gains much less. ResNet18 reaches 0.499 ± 0.144 on stripes and 0.874 on other-class backgrounds; the SE-CNN reaches 0.183 and 0.490. With the background removed, the SE-CNN control is even below its baseline (0.099 against 0.162). The gain therefore comes from breaking the link between background and label, not from training on cut-out leaves.

**Beyond controlled photos (Section 19).**
- **PlantDoc field photos** (244 photos: 63 healthy, 88 early blight, 91 leaf mold, 2 spider mites). Macro-F1 falls from at least 0.99 to 0.17–0.28 for every model. Early blight is recalled almost always (0.95–1.00 for the final models), healthy almost never (≤ 0.08) and leaf mold rarely (0.05–0.13). Most field photos thus appear to end up as early blight. The best score is SE-CNN + random backgrounds (0.282 ± 0.022): leaf mold recall reaches 0.47, but early blight drops to 0.80. Among the final models the ViT is best (0.215 ± 0.013). Field photos with several leaves, hands, soil and natural light differ in every respect, and their labels were checked only roughly. No model is usable in the field without field photos in training.

**Confidence and abstention (Section 20).**
- On the clean test the confidence flags the errors. Abstaining on the 5% least confident photos raises the accuracy to 0.999 (SE-CNN, ResNet18) and 1.000 (ViT). These 5% hold 85–100% of the errors, and the ECE is at most 0.01.
- Under the changes of Section 14, the models are **confidently wrong**. The ECE rises to 0.70 (SE-CNN, background removed), 0.52 (ResNet18 and SE-CNN, blur) and 0.35 (ViT, grayscale). Abstaining on 5% hardly changes the accuracy: 0.247 → 0.243 (SE-CNN, background removed), 0.420 → 0.422 (ResNet18, blur), 0.487 → 0.495 (ViT, grayscale). The confidence stays useful only where the model stays accurate (ViT under darkening and noise, ResNet18 under darkening, ECE ≤ 0.013). Abstention therefore cannot catch the failures that matter most, those under a shift of the input.

### 22.5 Limitations

- **Controlled photos only.** Disease is confounded with background, camera and light. The countermeasure addresses only the background; the PlantDoc results show that the other factors remain.
- **Three seeds on one fixed split.** The standard deviation covers training randomness, not the choice of test leaves. With 688 test photos, clean differences of 1–4 photos are not meaningful.
- **Saturated validation set.** Most recipes tie at 0.99–1.00, and the ablations ran on a single seed, so the choice between tied recipes is partly arbitrary.
- **Segmentation.** The background-removed photos come from PlantVillage's own segmentation, which also cuts away some leaf regions. The leaf masks of Sections 16–18 inherit these errors. The composed images (tiled background, mask edge) are unfamiliar in themselves, as the drop on same-class backgrounds shows.
- **Coverage.** Each change of Section 14 was tested at one strength only. The leaf share (Grad-CAM) and the countermeasure cover only the two CNNs, not the ViT. PlantDoc has rough labels and only 2 spider-mite photos.
- **Compute** was measured on one GPU; other devices change the absolute times, not necessarily the ratios.

### 22.6 Conclusion

- **Clean photos:** all three models reach at least 0.99 macro-F1. The ViT is slightly ahead, within the seed-to-seed variation of a few photos.
- **Robustness:** the models differ much more here than on clean photos. The ViT is the most robust to darkening, noise and blur. ResNet18 with random backgrounds is the most robust to background changes and keeps its clean score, at 45 MB, 3 ms per photo and one minute of training. The SE-CNN is the smallest and fastest model but the most dependent on the background and the most brittle.
- **Recommendation:** for photos taken under controlled conditions, ResNet18 trained with random backgrounds is the best trade-off between accuracy, background robustness and cost. Where lighting and cameras vary and compute allows it, the ViT is the safer choice.
- **Next steps:** no model is ready for the field. The next steps are field photos in training (for example the PlantDoc training split), photometric augmentation (blur, noise, darkening) together with random backgrounds, the countermeasure for the ViT, and evaluation on several splits.